In [1]:
import os
import gc
import logging
import pandas as pd
import numpy as np

try:
    %run setup_paths
except:
    %run notebooks/setup_paths
    

logging.basicConfig(
    level=logging.INFO,  # or DEBUG, WARNING, etc.
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

logging.info(f"current dir: {os.getcwd()}")


2026-09-20 10:26:04,834 - INFO - current dir: c:\Projects\CausalBenchmarkMTGClean


In [4]:
%run src/evaluation

df = pd.read_csv("results/baselines.comparison.csv")
sdf = pd.read_csv("results/baselines.comparison.split42.csv")

df, _ = add_null_baselines(df)
sdf, _= add_null_baselines(sdf)


In [5]:
METHOD_NAMES = {
    
    "dragonnet_cate_att": "DragonNet",

    "dr_att_gbm": "Doubly Robust",
    "cf_att": "Causal Forest",

    "dm_impute_t_att_lin": "T-Imputation (Linear)",
    "dm_impute_t_att_gbm": "T-Imputation (GBM)",
    

     "sipw_logit": "IPW (Logistic)",
    "sipw_gbm": "IPW (GBM)",

    "match_cov_k3": "Covariate Matching",
    "match_ps_logit_k3": "PS Matching (Logistic)",
    "match_ps_gbm_k3": "PS Matching (GBM)",
    

    "satt": "SATT",    
    "null_zero": "Null",
  
}

baseline_cols = list(METHOD_NAMES.keys())
res_cols = ["method","rmse","mae","ccc","accuracy95"]
res_sci_cols = ["method","rmse","rmse_sci","mae","mae_sci","ccc","ccc_sci","accuracy95","accuracy95_sci"]

In [7]:
res = evaluate_methods(df, baseline_cols, bounds=False)
res[res_sci_cols]

,method,rmse,rmse_sci,mae,mae_sci,ccc,ccc_sci,accuracy95,accuracy95_sci
0,cf_att,0.029778,0.001814,0.015973,0.000961,0.895989,0.013865,0.976015,0.003866
1,dragonnet_cate_att,0.030799,0.001825,0.016662,0.000991,0.890682,0.014672,0.970569,0.004034
2,dr_att_gbm,0.030940,0.001832,0.016401,0.000991,0.892054,0.014749,0.974486,0.004073
3,dm_impute_t_att_gbm,0.032068,0.001860,0.016968,0.001015,0.887035,0.015211,0.970473,0.004692
4,dm_impute_t_att_lin,0.032947,0.001938,0.017179,0.001036,0.882653,0.015931,0.969804,0.004613
5,sipw_gbm,0.039131,0.003178,0.019242,0.001306,0.849297,0.024383,0.952508,0.007967
6,match_ps_gbm_k3,0.039561,0.003382,0.020109,0.001291,0.838285,0.026495,0.939417,0.008347
7,match_cov_k3,0.041118,0.003395,0.020625,0.001338,0.834347,0.025883,0.937028,0.008131
8,sipw_logit,0.041118,0.003191,0.020126,0.001343,0.839978,0.024354,0.945533,0.008115
9,match_ps_logit_k3,0.041558,0.003232,0.020754,0.001349,0.832611,0.025885,0.933779,0.008511


In [8]:
res_sep = evaluate_methods(sdf, baseline_cols, bounds=False)
res_sep[res_sci_cols]

,method,rmse,rmse_sci,mae,mae_sci,ccc,ccc_sci,accuracy95,accuracy95_sci
0,cf_att,0.050930,0.002770,0.029707,0.001338,0.728495,0.031739,0.890779,0.006733
1,dr_att_gbm,0.052015,0.002917,0.030057,0.001365,0.728801,0.031970,0.889345,0.006840
2,dragonnet_cate_att,0.052383,0.002820,0.030493,0.001334,0.722423,0.031929,0.883516,0.006795
3,dm_impute_t_att_gbm,0.052819,0.002966,0.030405,0.001385,0.727227,0.032238,0.885428,0.006790
4,dm_impute_t_att_lin,0.053405,0.002986,0.030582,0.001389,0.724804,0.032600,0.885428,0.006699
5,match_ps_gbm_k3,0.059835,0.003923,0.033543,0.001618,0.664923,0.040361,0.857907,0.008028
6,sipw_gbm,0.060010,0.003679,0.033141,0.001674,0.684904,0.037580,0.869087,0.008204
7,match_cov_k3,0.061753,0.003591,0.034361,0.001612,0.671724,0.037473,0.853607,0.008475
8,sipw_logit,0.061784,0.003730,0.033828,0.001695,0.679290,0.038343,0.864978,0.007924
9,match_ps_logit_k3,0.061994,0.003799,0.034363,0.001615,0.668918,0.038815,0.849307,0.008512


In [9]:
%run src/sigmat
sigmat_sep = generate_significance_matrix(sdf, baseline_cols, one_sided=False, cluster_col="card_a")

In [10]:
nonsig = lambda x: x[~((x["rmse"] != '~') & (x["rmse"] == x["mae"]) & (x["mae"] == x["ccc"]) & (x["ccc"] == x["accuracy95"]))]

In [11]:
nonsig(sigmat_sep)

,method_a,method_b,n_pairs,n_clusters,rmse,mae,ccc,accuracy95
0,dragonnet_cate_att,dr_att_gbm,10515,197,~,<,<,<
2,dragonnet_cate_att,dm_impute_t_att_lin,10515,197,>,~,~,~
3,dragonnet_cate_att,dm_impute_t_att_gbm,10515,197,~,~,<,~
11,dr_att_gbm,cf_att,10515,197,<,<,~,~
21,cf_att,dm_impute_t_att_lin,10515,197,>,>,~,>
22,cf_att,dm_impute_t_att_gbm,10515,197,>,>,~,>
30,dm_impute_t_att_lin,dm_impute_t_att_gbm,10515,197,<,<,<,~
46,sipw_logit,match_cov_k3,10515,197,~,>,>,>
47,sipw_logit,match_ps_logit_k3,10515,197,~,>,>,>
48,sipw_logit,match_ps_gbm_k3,10515,197,<,~,>,>
